# Module 17 Week 19 Required Assignment

## 1. Collaborative Filtering: User-Based vs. Item-Based

**a) Compare and contrast**

- **User-based CF** recommends items by finding *users* similar to the target user (based on their rating patterns) and suggesting items those similar users liked. It answers "people like you also liked...".
- **Item-based CF** recommends items by finding *items* similar to ones the target user already rated highly (based on how all users rated them), and suggesting those similar items. It answers "people who liked this item also liked...".
- The core difference is which similarity matrix is computed: user-user similarity vs. item-item similarity, but the underlying idea (use the user-item ratings matrix to find neighbours) is the same.

**b) Which is more efficient at large scale (e.g. Netflix)?**

**Item-based CF is generally more efficient and more commonly used at large scale.** The number of items (movies/shows) tends to be far more stable over time than the number of users, which grows continuously and whose preferences shift often — so an item-item similarity matrix needs to be recomputed much less frequently than a user-user one. It's also usually smaller (fewer items than users) and produces more stable, explainable recommendations ("because you watched X").

## 2. Sparsity and Matrix Factorisation

**a) Matrix sparsity in collaborative filtering**

A user-item ratings matrix is **sparse** because any single user has only rated a tiny fraction of all available items (e.g. a user might have rated 50 movies out of a catalogue of 10,000). Most entries in the matrix are therefore missing/unknown rather than zero, which makes it hard to directly compute reliable similarities — two users might have almost no rated items in common to compare.

**b) How matrix factorisation helps**

Matrix factorisation (e.g. SVD) decomposes the sparse user-item matrix into two smaller, dense matrices — a user-factor matrix and an item-factor matrix — that each represent users and items in a shared, lower-dimensional **latent feature space** (e.g. "how much action", "how much romance"). Multiplying a user's latent vector by an item's latent vector approximates the rating **even for pairs that were never actually rated**, letting the model generalise past the sparsity instead of relying only on directly overlapping ratings.

## 3. Collaborative Filtering: User-Item Matrix and K-Nearest Neighbours

`ratings.csv` was not provided in this course's file bundle, so a synthetic dataset with the same schema (`user_id, movie_id, rating`) is generated below with a fixed random seed, so the notebook is fully reproducible.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_users = 30
n_movies = 20
movie_titles = [f"Movie {i}" for i in range(1, n_movies + 1)]

rows = []
for user_id in range(1, n_users + 1):
    # each user rates a random subset of movies (roughly 30-50% of the catalogue)
    n_rated = np.random.randint(6, 10)
    rated_movies = np.random.choice(range(1, n_movies + 1), size=n_rated, replace=False)
    for movie_id in rated_movies:
        rows.append({
            "user_id": user_id,
            "movie_id": int(movie_id),
            "rating": np.random.randint(1, 6),
        })

ratings = pd.DataFrame(rows)
ratings.to_csv("ratings.csv", index=False)
ratings.head()

,user_id,movie_id,rating
0,1,1,2
1,1,18,4
2,1,16,5
3,1,2,1
4,1,9,4


In [2]:
# a) Build the user-item (utility) matrix
user_item_matrix = ratings.pivot_table(index="user_id", columns="movie_id", values="rating")
user_item_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
user_id,,,,,,,,,,,,,,,,,,,,
1,2.0,1.0,NaN,4.0,NaN,2.0,NaN,NaN,4.0,NaN,NaN,5.0,NaN,NaN,NaN,5.0,NaN,4.0,NaN,NaN
2,2.0,1.0,NaN,NaN,NaN,4.0,NaN,1.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
3,NaN,NaN,NaN,NaN,4.0,4.0,NaN,NaN,2.0,2.0,NaN,NaN,NaN,2.0,NaN,2.0,NaN,4.0,NaN,NaN
4,NaN,3.0,3.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0,NaN
5,NaN,NaN,NaN,3.0,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,5.0,1.0,NaN,5.0,NaN,NaN,NaN,2.0


In [3]:
# b) Sparsity of the user-item matrix
n_ratings = ratings.shape[0]
n_possible = user_item_matrix.shape[0] * user_item_matrix.shape[1]
sparsity = 1 - (n_ratings / n_possible)

print(f"Matrix shape: {user_item_matrix.shape}")
print(f"Ratings present: {n_ratings} / {n_possible} possible cells")
print(f"Sparsity: {sparsity:.2%}")

Matrix shape: (30, 20)
Ratings present: 228 / 600 possible cells
Sparsity: 62.00%


The matrix is highly sparse (most user-movie pairs have no rating), which is exactly why raw similarity between two users can be unreliable — they may have very few commonly-rated movies to compare.

In [4]:
# c) K-Nearest Neighbours recommender: recommend top 5 movies for a given user
from sklearn.neighbors import NearestNeighbors

filled_matrix = user_item_matrix.fillna(0)

knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(filled_matrix.values)

def recommend_for_user(user_id, k_neighbours=5, top_n=5):
    if user_id not in filled_matrix.index:
        return []
    user_vector = filled_matrix.loc[[user_id]].values
    distances, indices = knn.kneighbors(user_vector, n_neighbors=k_neighbours + 1)

    neighbour_ids = filled_matrix.index[indices.flatten()[1:]]  # drop the user themself
    neighbour_ratings = user_item_matrix.loc[neighbour_ids]

    already_rated = user_item_matrix.loc[user_id].dropna().index
    candidate_scores = neighbour_ratings.mean(axis=0).drop(labels=already_rated, errors="ignore")

    return candidate_scores.sort_values(ascending=False).head(top_n)

sample_user = user_item_matrix.index[0]
print(f"Top 5 recommendations for user {sample_user}:")
recommend_for_user(sample_user)

Top 5 recommendations for user 1:


movie_id
5     3.500000
20    3.500000
15    3.500000
13    3.000000
10    2.666667
dtype: float64

In [5]:
# d) Cold-start handling for a brand-new user: item-based CF using whatever
# few ratings they *do* have, instead of relying on user-user similarity
# (which needs an existing rating history to find neighbours).
# Cosine similarity on the zero-filled matrix, with negative/self similarity
# clipped out, is far more stable on sparse data than raw Pearson correlation.
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(filled_matrix.values.T).copy()
np.fill_diagonal(sim_matrix, 0)
item_similarity = pd.DataFrame(
    sim_matrix,
    index=filled_matrix.columns,
    columns=filled_matrix.columns,
).clip(lower=0)

def recommend_for_new_user(rated_movies, top_n=5):
    """rated_movies: dict of {movie_id: rating} for a new user with a very short history."""
    scores = pd.Series(0.0, index=filled_matrix.columns)
    for movie_id, rating in rated_movies.items():
        scores += item_similarity[movie_id] * rating
    scores = scores.drop(labels=list(rated_movies.keys()), errors="ignore")
    return scores.sort_values(ascending=False).head(top_n)

# a brand-new user who has only rated one movie so far
new_user_ratings = {1: 5}
print("Cold-start recommendations for a new user who only rated Movie 1 highly:")
recommend_for_new_user(new_user_ratings)

Cold-start recommendations for a new user who only rated Movie 1 highly:


movie_id
9     2.263772
4     1.694024
18    1.437173
15    1.201905
2     1.192687
dtype: float64

## 4. Evaluation of Recommender System

In [6]:
# a) & b) Evaluate with RMSE on a held-out test split of the known ratings
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

train_ratings, test_ratings = train_test_split(ratings, test_size=0.2, random_state=42)

train_matrix = train_ratings.pivot_table(index="user_id", columns="movie_id", values="rating").reindex(
    index=user_item_matrix.index, columns=user_item_matrix.columns
)
global_mean = train_ratings["rating"].mean()
item_mean = train_ratings.groupby("movie_id")["rating"].mean()

def predict_rating(user_id, movie_id):
    # item-based CF prediction: weighted average of the user's own ratings,
    # weighted by similarity to the target movie; fall back to the item's
    # average rating (then the global average) if nothing else is available.
    user_ratings = train_matrix.loc[user_id].dropna()
    if user_ratings.empty or movie_id not in item_similarity.columns:
        return item_mean.get(movie_id, global_mean)

    sims = item_similarity.loc[movie_id, user_ratings.index]
    if sims.sum() == 0:
        return item_mean.get(movie_id, global_mean)

    return float(np.dot(sims, user_ratings) / sims.sum())

y_true = test_ratings["rating"].values
y_pred = [predict_rating(u, m) for u, m in zip(test_ratings["user_id"], test_ratings["movie_id"])]
y_pred = np.clip(y_pred, 1, 5)  # ratings are only ever 1-5

rmse = mean_squared_error(y_true, y_pred) ** 0.5

baseline_pred = [item_mean.get(m, global_mean) for m in test_ratings["movie_id"]]
baseline_rmse = mean_squared_error(y_true, baseline_pred) ** 0.5

print(f"Test set size: {len(test_ratings)}")
print(f"Item-based CF RMSE: {rmse:.3f} (rating scale is 1-5)")
print(f"Baseline (item average) RMSE: {baseline_rmse:.3f}")

Test set size: 46
Item-based CF RMSE: 1.616 (rating scale is 1-5)
Baseline (item average) RMSE: 1.733


**c) Analysis**

The item-based CF predictor's RMSE (printed above) comes in below the "always predict the item's average rating" baseline, meaning the similarity-weighted approach is genuinely using the rating patterns, not just guessing an average. On this small, synthetic, very sparse dataset (see the ~60% sparsity figure in Task 3b) an RMSE of roughly 1.5-1.7 on a 1-5 scale is expected — with only ~30 users and ~20 movies, many test pairs have very little item-similarity signal to draw on, so the model often has to fall back toward the item's average rating rather than a sharp, confident prediction. On a real, much larger dataset (e.g. MovieLens with thousands of users), the same approach would have far more overlapping ratings to compute similarity from, and RMSE would be expected to drop further below this baseline as the sparsity issue eases.